## Load the language model; set the GPU

In [ ]:
import os
from nnsight import LanguageModel

os.environ["CUDA_VISIBLE_DEVICES"] = "4"

model = LanguageModel(
    "Qwen/Qwen3-32B",
    attn_implementation="eager"
    device_map="cuda:0"
)
model.device

/disk/u/gio/.conda/envs/rle/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda', index=0)

## Load the problem

In [38]:
import json

dataset = []
with open("../data/hsp_step_accuracies.jsonl", 'r') as f:
    for line in f:
        dataset.append(json.loads(line))

problem2 = dataset[1]

essp_step_accuracies = problem2['essp_step_accuracies'][:-1]
hsp_step_accuracies = problem2['hsp_step_accuracies']

essp_index = problem2['first_essp_index']

cot_up_to_essp = "\n".join(problem2['steps'][:essp_index+1])

print(cot_up_to_essp)

Okay, so I need to figure out how many continuous paths there are from point A to point B along the segments of this figure without revisiting any of the six labeled points.
Let me start by visualizing the Asymptote figure they provided.
First, there's a rectangle with vertices at (0,0), (3,0), (3,2), and (0,2).
Then there are some additional lines: from (0,2) to (1,0), from (1,0) to (3,2), and from (0,2) to (1.5,3.5), which is point A.
Also, there's a line from (1.5,3.5) to (3,2).
The labeled points are A at the top, B at the bottom-left, C at the top-left, D at the top-right, E at the bottom-right, F at the middle-bottom.
So, the figure is a combination of a rectangle with a diagonal-like structure inside and a peak at the top.
Let me try to sketch this mentally.
The main rectangle is 3 units wide and 2 units tall.
Then from point C (0,2), there's a line going down to F (1,0), then up to D (3,2), and back to C.
Also, from C, there's a line going up to A (1.5,3.5), and from A to D.
So

## Build the prompt

In [39]:
tokenizer = model.tokenizer

SYSTEM_PROMPT = (
    "You are a helpful reasoning assistant. "
    "Output your final answer inside \\boxed{}."
)

def build_base_prompt(problem):
    """Build the base prompt for a problem."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


problem = problem2['problem']

base_prompt = build_base_prompt(problem)

prompt = f"{base_prompt}<think>\n{cot_up_to_essp}"
prompt += "\n</think>\nTherefore, the final answer is \\boxed{"

print(prompt)

<|im_start|>system
You are a helpful reasoning assistant. Output your final answer inside \boxed{}.<|im_end|>
<|im_start|>user
How many continuous paths from $A$ to $B$, along segments of the figure, do not revisit any of the six labeled points?

[asy]
draw((0,0)--(3,0)--(3,2)--(0,2)--(0,0)--cycle,linewidth(2));
draw((0,2)--(1,0)--(3,2)--(0,2)--cycle,linewidth(2));
draw((0,2)--(1.5,3.5)--(3,2),linewidth(2));

label("$A$",(1.5,3.5),N);
label("$B$",(0,0),SW);
label("$C$",(0,2),W);
label("$D$",(3,2),E);
label("$E$",(3,0),SE);
label("$F$",(1,0),S);
[/asy]<|im_end|>
<|im_start|>assistant
<think>
Okay, so I need to figure out how many continuous paths there are from point A to point B along the segments of this figure without revisiting any of the six labeled points.
Let me start by visualizing the Asymptote figure they provided.
First, there's a rectangle with vertices at (0,0), (3,0), (3,2), and (0,2).
Then there are some additional lines: from (0,2) to (1,0), from (1,0) to (3,2), and from

## Integrated Gradient Function

In [34]:
import torch
from tqdm import trange

def integrated_gradients(
    model: LanguageModel,
    input_text: str,
    target_token_id: int, # Which output token to attribute
    baseline_id: int | None = None,
    interpolation_steps: int = 50
):
    if baseline_id is None:
        baseline_id = model.tokenizer.pad_token_id or model.tokenizer.eos_token_id
        
    # Get the baseline embedding
    baseline_embed = model.model.embed_tokens.weight[baseline_id].detach()
    
    # Tokenize the full input
    token_ids = model.tokenizer.encode(input_text, add_special_tokens=False)
    tokens = model.tokenizer.tokenize(input_text)
    
    # Get all token embeddings: shape (seq_len, hidden_dim)
    token_embeds = model.model.embed_tokens.weight[token_ids].detach()
    
    # Baseline is repeated for each position: shape (seq_len, hidden_dim)
    baseline_embeds = baseline_embed.unsqueeze(0).expand_as(token_embeds)
    
    # Difference between input and baseline
    x_minus_baseline = token_embeds - baseline_embeds # (seq_len, hidden_dim)
    
    # Accumulate gradients across interpolation steps
    accumulated_grads = torch.zeros_like(token_embeds).to(device="cpu")
    print(accumulated_grads.device)
    
    for step in trange(1, interpolation_steps + 1):
        alpha = step / interpolation_steps
        interpolated_embeds = baseline_embeds + alpha * x_minus_baseline
        
        # Clear any lingering gradients
        model.zero_grad(set_to_none=True)
        
        with model.trace(input_text):
            # Move to correct device and add batch dimension INSIDE trace
            interpolated_embeds_traced = interpolated_embeds.unsqueeze(0).to(model.device).requires_grad_(True)
            
            # Override the embedding output
            model.model.embed_tokens.output = interpolated_embeds_traced
            
            # Get logits
            logits = model.output[0]
            target_logit = logits[0, -1, target_token_id]
            
            print(f"logits has nan: {logits.isnan().any()}")
            print(f"target_logit: {target_logit.item()}")
            print(f"interpolated_embeds has nan: {interpolated_embeds.isnan().any()}")
            
            # Compute gradients
            target_logit.backward()
            grad = interpolated_embeds_traced.grad.save()
            
            print(f"grad has nan: {grad.isnan().any()}, inf: {grad.isinf().any()}")
            
        print(f"{grad.device=}")
        
        accumulated_grads += grad.squeeze(0)
        
    # Average the gradients
    avg_grads = accumulated_grads / interpolation_steps
    
    print(avg_grads.device)
    
    print(x_minus_baseline.device)
    
    x_minus_baseline = x_minus_baseline.to(device='cpu')
    
    print(x_minus_baseline.device)
    
    # Integrated gradients = (x - x') * avg_grads
    ig_attributions = x_minus_baseline * avg_grads # (seq_len, hidden_dim)
    
    # sum across hidden dimension to get per-token attributino
    token_attributions = ig_attributions.sum(dim=-1) # (seq_len,)
    
    return {
        'tokens': tokens,
        'token_ids': token_ids,
        "attributions": token_attributions,
        "attributions_full": ig_attributions,
    }

## IG Saving Function

In [27]:
def to_json_safe(obj):
    if torch.is_tensor(obj):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [to_json_safe(x) for x in obj]
    else:
        return obj

## Helper Methods

In [28]:
def extract_cot_minus_answer(full_cot):
    # Calculate the index of the last character of "\\boxed{"
    index_of_boxed = full_cot.rfind("\\boxed{", )
    index_of_answer = index_of_boxed + +len("\\boxed{")
    return full_cot[:index_of_answer]

def get_target_token_id(full_cot, cot_minus_answer):
    """Get the id of the first token of the final answer."""
    token_ids = model.tokenizer.encode(full_cot, add_special_tokens=False)
    prefix_tokens = model.tokenizer.encode(cot_minus_answer, add_special_tokens=False)
    target_token_id = token_ids[len(prefix_tokens)]
    return target_token_id

## Sanity Check

In [40]:
answer = problem2['answer']
full_cot = problem2['full_cot']
print(answer)

cot_minus_answer = extract_cot_minus_answer(full_cot)

target_id = get_target_token_id(full_cot, cot_minus_answer)
print(target_id)
print(model.tokenizer.decode(target_id))

10
16
1


In [41]:
prompt

'<|im_start|>system\nYou are a helpful reasoning assistant. Output your final answer inside \\boxed{}.<|im_end|>\n<|im_start|>user\nHow many continuous paths from $A$ to $B$, along segments of the figure, do not revisit any of the six labeled points?\n\n[asy]\ndraw((0,0)--(3,0)--(3,2)--(0,2)--(0,0)--cycle,linewidth(2));\ndraw((0,2)--(1,0)--(3,2)--(0,2)--cycle,linewidth(2));\ndraw((0,2)--(1.5,3.5)--(3,2),linewidth(2));\n\nlabel("$A$",(1.5,3.5),N);\nlabel("$B$",(0,0),SW);\nlabel("$C$",(0,2),W);\nlabel("$D$",(3,2),E);\nlabel("$E$",(3,0),SE);\nlabel("$F$",(1,0),S);\n[/asy]<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, so I need to figure out how many continuous paths there are from point A to point B along the segments of this figure without revisiting any of the six labeled points.\nLet me start by visualizing the Asymptote figure they provided.\nFirst, there\'s a rectangle with vertices at (0,0), (3,0), (3,2), and (0,2).\nThen there are some additional lines: from (0,2) to (1,0), from

## Compute IG

In [42]:
result = integrated_gradients(
    model=model,
    input_text=prompt,
    target_token_id=target_id,
    interpolation_steps=20
)

for token, attr in zip(result["tokens"], result["attributions"]):
    print(f"{token:>10}: {attr.item():+.4f}")

cpu


  0%|          | 0/20 [00:00<?, ?it/s]

logits has nan: False
target_logit: 5.973130702972412
interpolated_embeds has nan: False
grad has nan: False, inf: False


  5%|▌         | 1/20 [07:08<2:15:44, 428.67s/it]

grad.device=device(type='cpu')
logits has nan: False
target_logit: 5.932710647583008
interpolated_embeds has nan: False


...

grad has nan: False, inf: False
grad.device=device(type='cpu')
logits has nan: False
target_logit: 39.0639533996582
interpolated_embeds has nan: False


 95%|█████████▌| 19/20 [2:30:34<07:51, 471.13s/it]

grad has nan: False, inf: False
grad.device=device(type='cpu')
logits has nan: False
target_logit: 39.10830307006836
interpolated_embeds has nan: False


100%|██████████| 20/20 [2:37:51<00:00, 473.58s/it]

grad has nan: False, inf: False
grad.device=device(type='cpu')
cpu
cpu
cpu
<|im_start|>: +3.4524
    system: -9.0920
         Ċ: -2.5129
       You: -0.8652
      Ġare: -0.2570
        Ġa: -0.3489
  Ġhelpful: +0.0668
Ġreasoning: -0.1231
Ġassistant: +0.6797
         .: +1.3178
   ĠOutput: +0.9088
     Ġyour: -0.0287
    Ġfinal: +0.4104
   Ġanswer: -0.0779
   Ġinside: -0.4071
        Ġ\: -0.2191
     boxed: -2.2593
       {}.: -0.0229
<|im_end|>: +0.0280
         Ċ: -0.5550
<|im_start|>: -0.1827
      user: +0.1104
         Ċ: -0.1606
       How: -1.3031
     Ġmany: -1.7969
Ġcontinuous: -0.5502
    Ġpaths: -3.2962
     Ġfrom: -0.2936
        Ġ$: +0.4375
         A: -0.1030
         $: +1.2961
       Ġto: +0.6084
        Ġ$: +0.9470
         B: +0.1504
        $,: +0.9899
    Ġalong: +0.4196
 Ġsegments: -0.2776
       Ġof: +1.1653
      Ġthe: +2.9888
   Ġfigure: +1.5826
         ,: +0.7848
       Ġdo: +0.3217
      Ġnot: +0.2556
  Ġrevisit: -0.5588
      Ġany: -0.1048
       Ġof: +0.1988


## Save Result

In [43]:
with open('../data/attributions/result2.1.json', 'w') as f:
    json.dump(to_json_safe(result), f, indent=2)

## Load the problem

In [44]:
import json

dataset = []
with open("../data/hsp_step_accuracies.jsonl", 'r') as f:
    for line in f:
        dataset.append(json.loads(line))

problem3 = dataset[2]

essp_step_accuracies = problem3['essp_step_accuracies'][:-1]
hsp_step_accuracies = problem3['hsp_step_accuracies']

essp_index = problem3['first_essp_index']

cot_up_to_essp = "\n".join(problem3['steps'][:essp_index+1])

print(cot_up_to_essp)

Okay, so I need to find the minimum possible value of |x₁ + x₂ + ...
+ x₂₀₀₆| given that x₀ = 0 and |x_k| = |x_{k-1} + 3| for all integers k ≥ 1.
Hmm, let me try to understand the problem first.
Starting with x₀ = 0.
Then for each k ≥ 1, the absolute value of x_k is equal to the absolute value of x_{k-1} + 3.
So, for each term, we have two possibilities: x_k = x_{k-1} + 3 or x_k = -(x_{k-1} + 3).
Because when you take absolute value on both sides, squaring both sides would give x_k² = (x_{k-1} + 3)², so x_k can be either positive or negative of that expression.
Therefore, at each step, we can choose the sign of x_k.
The problem is to choose these signs in such a way that the absolute value of the sum from x₁ to x₂₀₀₆ is minimized.
So, the challenge is to figure out a sequence of signs (either + or -) for each term such that when you add up all these terms, the total is as close to zero as possible.
Since we're taking absolute value at the end, we want the sum to be as close to zero as 

## Build the prompt

In [45]:
tokenizer = model.tokenizer

SYSTEM_PROMPT = (
    "You are a helpful reasoning assistant. "
    "Output your final answer inside \\boxed{}."
)

def build_base_prompt(problem):
    """Build the base prompt for a problem."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


problem = problem3['problem']

base_prompt = build_base_prompt(problem)

prompt = f"{base_prompt}<think>\n{cot_up_to_essp}"
prompt += "\n</think>\nTherefore, the final answer is \\boxed{"

print(prompt)

<|im_start|>system
You are a helpful reasoning assistant. Output your final answer inside \boxed{}.<|im_end|>
<|im_start|>user
Given that a sequence satisfies $x_0=0$ and $|x_k|=|x_{k-1}+3|$ for all integers $k\ge1$, find the minimum possible value of $|x_1+x_2+\cdots+x_{2006}|$.<|im_end|>
<|im_start|>assistant
<think>
Okay, so I need to find the minimum possible value of |x₁ + x₂ + ...
+ x₂₀₀₆| given that x₀ = 0 and |x_k| = |x_{k-1} + 3| for all integers k ≥ 1.
Hmm, let me try to understand the problem first.
Starting with x₀ = 0.
Then for each k ≥ 1, the absolute value of x_k is equal to the absolute value of x_{k-1} + 3.
So, for each term, we have two possibilities: x_k = x_{k-1} + 3 or x_k = -(x_{k-1} + 3).
Because when you take absolute value on both sides, squaring both sides would give x_k² = (x_{k-1} + 3)², so x_k can be either positive or negative of that expression.
Therefore, at each step, we can choose the sign of x_k.
The problem is to choose these signs in such a way that

## Integrated Gradient Function

In [ ]:
import torch
from tqdm import trange

def integrated_gradients(
    model: LanguageModel,
    input_text: str,
    target_token_id: int, # Which output token to attribute
    baseline_id: int | None = None,
    interpolation_steps: int = 50
):
    if baseline_id is None:
        baseline_id = model.tokenizer.pad_token_id or model.tokenizer.eos_token_id
        
    # Get the baseline embedding
    baseline_embed = model.model.embed_tokens.weight[baseline_id].detach()
    
    # Tokenize the full input
    token_ids = model.tokenizer.encode(input_text, add_special_tokens=False)
    tokens = model.tokenizer.tokenize(input_text)
    
    # Get all token embeddings: shape (seq_len, hidden_dim)
    token_embeds = model.model.embed_tokens.weight[token_ids].detach()
    
    # Baseline is repeated for each position: shape (seq_len, hidden_dim)
    baseline_embeds = baseline_embed.unsqueeze(0).expand_as(token_embeds)
    
    # Difference between input and baseline
    x_minus_baseline = token_embeds - baseline_embeds # (seq_len, hidden_dim)
    
    # Accumulate gradients across interpolation steps
    accumulated_grads = torch.zeros_like(token_embeds).to(device="cpu")
    print(accumulated_grads.device)
    
    for step in trange(1, interpolation_steps + 1):
        alpha = step / interpolation_steps
        interpolated_embeds = baseline_embeds + alpha * x_minus_baseline
        
        # Clear any lingering gradients
        model.zero_grad(set_to_none=True)
        
        with model.trace(input_text):
            # Move to correct device and add batch dimension INSIDE trace
            interpolated_embeds_traced = interpolated_embeds.unsqueeze(0).to(model.device).requires_grad_(True)
            
            # Override the embedding output
            model.model.embed_tokens.output = interpolated_embeds_traced
            
            # Get logits
            logits = model.output[0]
            target_logit = logits[0, -1, target_token_id]
            
            print(f"logits has nan: {logits.isnan().any()}")
            print(f"target_logit: {target_logit.item()}")
            print(f"interpolated_embeds has nan: {interpolated_embeds.isnan().any()}")
            
            # Compute gradients
            target_logit.backward()
            grad = interpolated_embeds_traced.grad.save()
            
            print(f"grad has nan: {grad.isnan().any()}, inf: {grad.isinf().any()}")
            
        print(f"{grad.device=}")
        
        accumulated_grads += grad.squeeze(0)
        
    # Average the gradients
    avg_grads = accumulated_grads / interpolation_steps
    
    print(avg_grads.device)
    
    print(x_minus_baseline.device)
    
    x_minus_baseline = x_minus_baseline.to(device='cpu')
    
    print(x_minus_baseline.device)
    
    # Integrated gradients = (x - x') * avg_grads
    ig_attributions = x_minus_baseline * avg_grads # (seq_len, hidden_dim)
    
    # sum across hidden dimension to get per-token attributino
    token_attributions = ig_attributions.sum(dim=-1) # (seq_len,)
    
    return {
        'tokens': tokens,
        'token_ids': token_ids,
        "attributions": token_attributions,
        "attributions_full": ig_attributions,
    }

## IG Saving Function

In [ ]:
def to_json_safe(obj):
    if torch.is_tensor(obj):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [to_json_safe(x) for x in obj]
    else:
        return obj

## Helper Methods

In [ ]:
def extract_cot_minus_answer(full_cot):
    # Calculate the index of the last character of "\\boxed{"
    index_of_boxed = full_cot.rfind("\\boxed{", )
    index_of_answer = index_of_boxed + +len("\\boxed{")
    return full_cot[:index_of_answer]

def get_target_token_id(full_cot, cot_minus_answer):
    """Get the id of the first token of the final answer."""
    token_ids = model.tokenizer.encode(full_cot, add_special_tokens=False)
    prefix_tokens = model.tokenizer.encode(cot_minus_answer, add_special_tokens=False)
    target_token_id = token_ids[len(prefix_tokens)]
    return target_token_id

## Sanity Check

In [46]:
answer = problem3['answer']
full_cot = problem3['full_cot']
print(answer)

cot_minus_answer = extract_cot_minus_answer(full_cot)

target_id = get_target_token_id(full_cot, cot_minus_answer)
print(target_id)
print(model.tokenizer.decode(target_id))

27
17
2


In [ ]:
prompt

'<|im_start|>system\nYou are a helpful reasoning assistant. Output your final answer inside \\boxed{}.<|im_end|>\n<|im_start|>user\nHow many continuous paths from $A$ to $B$, along segments of the figure, do not revisit any of the six labeled points?\n\n[asy]\ndraw((0,0)--(3,0)--(3,2)--(0,2)--(0,0)--cycle,linewidth(2));\ndraw((0,2)--(1,0)--(3,2)--(0,2)--cycle,linewidth(2));\ndraw((0,2)--(1.5,3.5)--(3,2),linewidth(2));\n\nlabel("$A$",(1.5,3.5),N);\nlabel("$B$",(0,0),SW);\nlabel("$C$",(0,2),W);\nlabel("$D$",(3,2),E);\nlabel("$E$",(3,0),SE);\nlabel("$F$",(1,0),S);\n[/asy]<|im_end|>\n<|im_start|>assistant\n<think>\nOkay, so I need to figure out how many continuous paths there are from point A to point B along the segments of this figure without revisiting any of the six labeled points.\nLet me start by visualizing the Asymptote figure they provided.\nFirst, there\'s a rectangle with vertices at (0,0), (3,0), (3,2), and (0,2).\nThen there are some additional lines: from (0,2) to (1,0), from

## Compute IG

In [47]:
result = integrated_gradients(
    model=model,
    input_text=prompt,
    target_token_id=target_id,
    interpolation_steps=20
)

for token, attr in zip(result["tokens"], result["attributions"]):
    print(f"{token:>10}: {attr.item():+.4f}")

cpu


  0%|          | 0/20 [00:00<?, ?it/s]

logits has nan: False
target_logit: 5.713953971862793
interpolated_embeds has nan: False
grad has nan: False, inf: False


  5%|▌         | 1/20 [15:34<4:55:52, 934.35s/it]

grad.device=device(type='cpu')
logits has nan: False
target_logit: 5.681481838226318
interpolated_embeds has nan: False
grad has nan: False, inf: False


...

 95%|█████████▌| 19/20 [4:58:44<15:40, 940.98s/it]

grad.device=device(type='cpu')
logits has nan: False
target_logit: 34.0462760925293
interpolated_embeds has nan: False
grad has nan: False, inf: False


100%|██████████| 20/20 [5:14:24<00:00, 943.20s/it]

grad.device=device(type='cpu')
cpu
cpu
cpu


<|im_start|>: -4.8963
    system: -8.4227
         Ċ: -1.7533
       You: +0.4352
      Ġare: +0.2428
        Ġa: +0.5068
  Ġhelpful: +0.8906
Ġreasoning: -0.0395
Ġassistant: +0.6237
         .: +1.5174
   ĠOutput: -0.1911
     Ġyour: -0.2598
    Ġfinal: -0.3663
   Ġanswer: -1.4276
   Ġinside: -0.1000
        Ġ\: -0.6368
     boxed: -1.1456
       {}.: -0.6563
<|im_end|>: -0.0201
         Ċ: -0.9445
<|im_start|>: -0.1310
      user: -0.4721
         Ċ: -0.2497
     Given: -1.8536
     Ġthat: -0.4659
        Ġa: -0.4605
 Ġsequence: -0.6604
Ġsatisfies: -0.6942
        Ġ$: -0.4713
         x: -0.3114
         _: +0.0698
         0: -0.2016
         =: -0.0887
         0: +0.1677
         $: -0.5570
      Ġand: -0.7760
        Ġ$: -1.3544
         |: -0.5794
         x: -0.2831
        _k: -0.1340
        |=: -0.4807
         |: -0.7526
         x: +1.5902
        _{: +13.1735
         k: -0.0769
         -: -0.0305
         1: -0.2274
         }: -0.4573
         +: -0.0557
         3: +0.

## Save Result

In [ ]:
with open('../data/attributions/result3.1.json', 'w') as f:
    json.dump(to_json_safe(result), f, indent=2)

: 